In [3]:
import fastf1
import pandas as pd
import numpy as np

# Enable FastF1 cache
fastf1.Cache.enable_cache('D:\\Programming\\Python\\AI\\F1 Fantasy\\cache')

In [23]:
round_number = 15
year = 2026

session = fastf1.get_session(year, round_number, 'r')
session.load(telemetry=False)

core           INFO 	Loading data for Azerbaijan Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data fo

In [24]:
laps = session.laps.copy()
laps = laps.pick_accurate().pick_wo_box().pick_track_status('1').copy()
laps = laps[laps['LapTime'].notna()]
laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()
laps['StartTimeSeconds'] = laps['LapStartTime'].dt.total_seconds()

laps['StintLap'] = laps.groupby(['Driver', 'Stint']).cumcount() + 1
laps = laps[laps['StintLap'] > 1]

laps.Team

2           McLaren
3           McLaren
4           McLaren
5           McLaren
6           McLaren
           ...     
974    Haas F1 Team
975    Haas F1 Team
976    Haas F1 Team
977    Haas F1 Team
978    Haas F1 Team
Name: Team, Length: 662, dtype: object

In [25]:
# Within each stint, calculate rolling minimum and discard lap times more than thresh seconds above it
def drop_traffic(g, thresh=1.3):
    baseTime = g['LapTimeSeconds'].rolling(5, center=True, min_periods=2).min()
    return g[g['LapTimeSeconds'] < baseTime + thresh]

laps = laps.groupby(['Driver', 'Stint'], group_keys=False).apply(drop_traffic)
laps.head()

C:\Users\nanub\AppData\Local\Temp\ipykernel_15084\693568425.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  laps = laps.groupby(['Driver', 'Stint'], group_keys=False).apply(drop_traffic)


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,LapTimeSeconds,StartTimeSeconds,StintLap
255,0 days 01:02:35.163000,ALB,23,0 days 00:01:50.388000,3.0,1.0,NaT,NaT,0 days 00:00:39.788000,0 days 00:00:45.391000,...,NaT,1,12.0,False,,False,True,110.388,3644.775,2
256,0 days 01:04:25.225000,ALB,23,0 days 00:01:50.062000,4.0,1.0,NaT,NaT,0 days 00:00:39.449000,0 days 00:00:45.308000,...,NaT,1,12.0,False,,False,True,110.062,3755.163,3
257,0 days 01:06:14.776000,ALB,23,0 days 00:01:49.551000,5.0,1.0,NaT,NaT,0 days 00:00:39.285000,0 days 00:00:44.947000,...,NaT,1,12.0,False,,False,True,109.551,3865.225,4
258,0 days 01:08:04.608000,ALB,23,0 days 00:01:49.832000,6.0,1.0,NaT,NaT,0 days 00:00:39.458000,0 days 00:00:45.361000,...,NaT,1,12.0,False,,False,True,109.832,3974.776,5
259,0 days 01:09:54.308000,ALB,23,0 days 00:01:49.700000,7.0,1.0,NaT,NaT,0 days 00:00:39.469000,0 days 00:00:45.089000,...,NaT,1,12.0,False,,False,True,109.700,4084.608,6


In [26]:
# Fuel per-circuit estimate
K_FUEL = 0.03 * (laps['LapTimeSeconds'].median() / 90)
# Assume about 1.7kg burned per lap
BURN = 1.7

# Relative fuel load, highest at the start of a long run
# Assumes teams start long runs at similar loads
# although the assumption is essentially always wrong, it averages across a session and gives a smaller error than when fuel is just ignored
laps['Fuel_kg'] = (laps.groupby(['Driver', 'Stint'])['StintLap'].transform('max') - laps['StintLap'] * BURN)

In [27]:
# Fit a single regression across every clean long run lap in the session and let the driver coefficients fall out
import statsmodels.formula.api as smf

laps['TyreLifeFloat'] = laps['TyreLife'].astype(float)
laps['evo'] = (laps['StartTimeSeconds'] - laps['StartTimeSeconds'].min()) / 3600

# This regression is fitting every effect at once, using all the laps in the session together
# This makes it so that each coefficient is estimated controlling for all the others at the same time
# The formula can be read as:
# Predict this lap's time as combination of the driver that did it, how much fuel there was, which compound, age of the tyre, 
model = smf.rlm(
    'LapTimeSeconds ~ 0 + C(Driver) + Fuel_kg + C(Compound) + C(Compound):TyreLifeFloat + evo',
    data=laps
).fit()

rpace_raw = model.params.filter(like='C(Driver)')
rpace_raw.index = rpace_raw.index.str.extract(r'\[(.*)\]')[0].values

# Seconds off the fastest driver
rpace_raw -= rpace_raw.min()
rpace_raw

# Convert to fractional units for the simulator
rpace = rpace_raw / laps['LapTimeSeconds'].median()
rpace

ALB    0.018013
ALO    0.040747
ANT    0.004790
BEA    0.016920
BOR    0.018398
BOT    0.038839
COL    0.013077
GAS    0.009129
HAD    0.005339
HAM    0.009736
HUL    0.019463
LAW    0.018208
LEC    0.006850
LIN    0.017771
NOR    0.007468
OCO    0.017524
PER    0.022492
PIA    0.007660
RUS    0.000000
SAI    0.019098
STR    0.036969
VER    0.001796
dtype: float64

In [28]:
# se: standard error (estimated standard deviation of the coefficent estimate)
# means the uncertainty in the estimate of the mean
se_raw = model.bse.filter(like='C(Driver)')
se_raw.index = se_raw.index.str.extract(r'\[(.*)\]')[0].values
rpace_se_frac = se_raw / laps['LapTimeSeconds'].median()

# Inflate for risk that the regression can't see
# ex. driver who only ran one compound, or whose longest stint was only 4 laps
n_laps = laps.groupby('Driver').size()
n_compounds = laps.groupby('Driver')['Compound'].nunique()
rpace_se_frac *= np.where(n_compounds < 2, 1.3, 1.0)
rpace_se_frac *= np.clip(8 / n_laps.clip(lower=1), 1.0, 2.5)
rpace_se_frac

ALB    0.001997
ALO    0.001670
ANT    0.001739
BEA    0.001557
BOR    0.001564
BOT    0.001553
COL    0.002048
GAS    0.002105
HAD    0.001551
HAM    0.001551
HUL    0.001534
LAW    0.001886
LEC    0.001551
LIN    0.001557
NOR    0.002118
OCO    0.001531
PER    0.001558
PIA    0.001551
RUS    0.001683
SAI    0.001348
STR    0.002987
VER    0.001574
dtype: float64

In [29]:
round_result = pd.DataFrame({
    'Driver': rpace_raw.index,
    'Team': [laps[laps.Driver == d].Team.values[0] for d in rpace_raw.index],
    'Round': round_number,
    'Year': year,
    'rpace_frac': rpace.values,
    'rpace_se_frac': rpace_se_frac.values,
    'n_laps': laps.groupby('Driver').size().reindex(rpace_raw.index).values,
})
round_result

,Driver,Team,Round,Year,rpace_frac,rpace_se_frac,n_laps
0,ALB,Williams,15,2026,0.018013,0.001997,23
1,ALO,Aston Martin,15,2026,0.040747,0.001670,15
2,ANT,Mercedes,15,2026,0.004790,0.001739,29
3,BEA,Haas F1 Team,15,2026,0.016920,0.001557,32
4,BOR,Audi,15,2026,0.018398,0.001564,31
5,BOT,Cadillac,15,2026,0.038839,0.001553,28
6,COL,Alpine,15,2026,0.013077,0.002048,25
7,GAS,Alpine,15,2026,0.009129,0.002105,26
8,HAD,Red Bull Racing,15,2026,0.005339,0.001551,34
9,HAM,Ferrari,15,2026,0.009736,0.001551,35


In [30]:
race_laps_archive = pd.read_csv('data/race_archive.csv')
race_laps_archive = pd.concat([race_laps_archive, round_result], ignore_index=True)
race_laps_archive.to_csv('data/race_archive.csv', index=False)

In [4]:
def source_race_form(driver, archive, current_round, half_life=4):
    rows = archive[archive['Driver'] == driver]
    if rows.empty:
        return 0.0, np.inf

    ests, ses, weights = [], [], []
    for round, group in rows.groupby('Round'):
        age = current_round = round
        if age <= 0:
            continue # skip round if it's somehow in the archive

        age_w = 0.5 ** (age / half_life)
        ests.append(group['rpace_frac'].iloc[0])
        ses.append(group['rpace_se_frac'].iloc[0])
        weights.append(age_w)

    if not weights or sum(weights) == 0:
        return 0.0, np.inf

    ests, ses, weights = map(np.array, (ests, ses, weights))
    w = weights / ses**2          # combines recency weight with inverse-variance weight

    # mu: estimate of driver's fractional pace delta
    # ex. 0.0014 means 1.4 thousandths slower per lap than reference, which at a 90s lap is about 0.13s off the fastest
    mu = np.sum(w * ests) / np.sum(w) # 

    # sigma: estimate of how much mu can be trusted
    # smaller sigma = more confident (more clean laps, consistent), larger sigma = less confident
    sigma = np.sqrt(1.0 / np.sum(w))
    return mu, sigma

In [5]:
race_laps_archive = pd.read_csv('data/race_archive.csv')
source_race_form('HAM', race_laps_archive, 11)

(np.float64(0.005127920503817842), np.float64(0.0006256045746985599))

In [6]:
current_round = 16

historical_rpace = {}
historical_sigma = {}
for driver in race_laps_archive['Driver'].unique():
    mu, sigma = source_race_form(driver, race_laps_archive, current_round)
    historical_rpace[driver] = mu
    historical_sigma[driver] = sigma

historical_rpace = pd.Series(historical_rpace)
historical_sigma = pd.Series(historical_sigma)

historical_rpace = pd.concat([historical_rpace, historical_sigma], axis=1, keys=['mu', 'sigma'])
historical_rpace

,mu,sigma
ALB,0.028382,0.000659
ALO,0.040136,0.000820
ANT,0.000638,0.000665
BEA,0.022497,0.000673
BOR,0.021661,0.000652
BOT,0.041238,0.000813
COL,0.021858,0.000664
GAS,0.017952,0.000680
HAD,0.014572,0.000785
HAM,0.005128,0.000626


In [7]:
historical_rpace.to_csv("historical_rpace.csv")